In [112]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression


In [113]:
# 1. Load the data from the CSV file into a pandas DataFrame
csv_file_path = '/Users/tvo/Documents/portfolio/college-notes/CSC 180/homeprices3.csv'

df = pd.read_csv(csv_file_path)
print("Original Data:")
print(df.head())
print("-" * 30)

Original Data:
              town  area   price
0  monroe township  2600  550000
1  monroe township  3000  565000
2  monroe township  3200  610000
3  monroe township  3600  680000
4  monroe township  4000  725000
------------------------------


In [114]:
# Check for Missing Data
missing_cols = df.isna().any()
if missing_cols.any():
    cols_with_missing = missing_cols[missing_cols].index
    for col in cols_with_missing:
        print(f"Data Missing in {col}")
else:
    print("No Missing Data in Any Column")

No Missing Data in Any Column


In [115]:
# Split Training and Test Data
target = 'price'
features = [col for col in df.columns if col != target]

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [116]:
# RandomForestRegressor w/ Dummy Variable

numeric_features = ['area']
categorical_features = ['town']

transformer_steps = [
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
]
preprocessor = ColumnTransformer(transformers=transformer_steps)

pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor()) 
])

param_dist = {
    'model__n_estimators': [50, 100, 150, 200, 300],
    'model__max_depth': [5, 10, 15, 20, None],
    'model__min_samples_leaf': [1, 2, 4, 6],
    'model__max_features': ['sqrt', 1.0]
}

random_search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist, 
    n_iter=20,
    cv=3,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

best_params_from_random = random_search.best_params_

param_grid_focused = {
    'model__n_estimators': [best_params_from_random['model__n_estimators']], 
    'model__max_depth': [8, 10, 12], 
    'model__min_samples_leaf': [1, 2, 3], 
    'model__max_features': [best_params_from_random['model__max_features']] 
}

grid_search = GridSearchCV(estimator=pipe, param_grid=param_grid_focused, cv=3, n_jobs=-1)

grid_search.fit(X_train, y_train)

,estimator,Pipeline(step...Regressor())])
,param_grid,"{'model__max_depth': [8, 10, ...], 'model__max_features': [1.0], 'model__min_samples_leaf': [1, 2, ...], 'model__n_estimators': [150]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [117]:
print("--- Model Results ---")
best_pipeline = grid_search.best_estimator_
model = best_pipeline.named_steps['model']
preprocessor = best_pipeline.named_steps['preprocessor']
importances = model.feature_importances_
feature_names = preprocessor.get_feature_names_out()

print("Feature Importances:")
for name, importance in zip(feature_names, importances):
    print(f"{name}: {importance:.4f}")

print("\n--- GridSearch Stats ---")
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
print(f"Test set score: {grid_search.score(X_test, y_test):.4f}")

--- Model Results ---
Feature Importances:
num__area: 0.7811
cat__town_monroe township: 0.0806
cat__town_robinsville: 0.0255
cat__town_west windsor: 0.1128

--- GridSearch Stats ---
Best parameters found: {'model__max_depth': 12, 'model__max_features': 1.0, 'model__min_samples_leaf': 1, 'model__n_estimators': 150}
Best cross-validation score: -4.4287
Test set score: 0.1013


In [118]:
# RandomForestRegressor w/out Dummy Variable
numeric_features = ['area']
categorical_features = ['town']

transformer_steps = [
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
]

preprocessor = ColumnTransformer(transformers=transformer_steps)

pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor()) 
])

param_dist = {
    'model__n_estimators': [50, 100, 150, 200, 300],
    'model__max_depth': [5, 10, 15, 20, None],
    'model__min_samples_leaf': [1, 2, 4, 6],
    'model__max_features': ['sqrt', 1.0]
}

random_search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist, 
    n_iter=20,
    cv=3,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

best_params_from_random = random_search.best_params_

param_grid_focused = {
    'model__n_estimators': [best_params_from_random['model__n_estimators']], 
    'model__max_depth': [8, 10, 12], 
    'model__min_samples_leaf': [1, 2, 3], 
    'model__max_features': [best_params_from_random['model__max_features']] 
}

grid_search_dummy = GridSearchCV(estimator=pipe, param_grid=param_grid_focused, cv=3, n_jobs=-1)

grid_search_dummy.fit(X_train, y_train)

,estimator,Pipeline(step...Regressor())])
,param_grid,"{'model__max_depth': [8, 10, ...], 'model__max_features': [1.0], 'model__min_samples_leaf': [1, 2, ...], 'model__n_estimators': [150]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [119]:
print("--- Model Results DUMMY---")
best_pipeline = grid_search_dummy.best_estimator_
model = best_pipeline.named_steps['model']
preprocessor = best_pipeline.named_steps['preprocessor']
importances = model.feature_importances_
feature_names = preprocessor.get_feature_names_out()

print("Feature Importances:")
for name, importance in zip(feature_names, importances):
    print(f"{name}: {importance:.4f}")

print("\n--- GridSearch Stats ---")
print(f"Best parameters found: {grid_search_dummy.best_params_}")
print(f"Best cross-validation score: {grid_search_dummy.best_score_:.4f}")
print(f"Test set score: {grid_search_dummy.score(X_test, y_test):.4f}")

--- Model Results DUMMY---
Feature Importances:
num__area: 0.8130
cat__town_robinsville: 0.0367
cat__town_west windsor: 0.1503

--- GridSearch Stats ---
Best parameters found: {'model__max_depth': 8, 'model__max_features': 1.0, 'model__min_samples_leaf': 1, 'model__n_estimators': 150}
Best cross-validation score: -3.5891
Test set score: 0.0814


In [120]:
# LinearRegrsssion w/out Dummy Variable
numeric_features = ['area']
categorical_features = ['town']

transformer_steps = [
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
]

preprocessor = ColumnTransformer(transformers=transformer_steps)

pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression()) 
])

pipe.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [121]:
print(f"Test set score: {pipe.score(X_test, y_test):.4f}")

model = pipe.named_steps['model']
feature_names = pipe.named_steps['preprocessor'].get_feature_names_out()

print("\nCoefficients:")
for name, coef in zip(feature_names, model.coef_):
    print(f"{name}: {coef:.2f}")

Test set score: 0.8005

Coefficients:
num__area: 47485.93
cat__town_robinsville: 7333.33
cat__town_west windsor: 15000.00


In [122]:
# LinearRegrsssion w/ Dummy Variable
numeric_features = ['area']
categorical_features = ['town']

transformer_steps = [
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
]

preprocessor = ColumnTransformer(transformers=transformer_steps)

pipe_dummy = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression()) 
])

pipe_dummy.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [123]:
print(f"Test set score: {pipe_dummy.score(X_test, y_test):.4f}")

model = pipe_dummy.named_steps['model']
feature_names = pipe_dummy.named_steps['preprocessor'].get_feature_names_out()

print("\nCoefficients:")
for name, coef in zip(feature_names, model.coef_):
    print(f"{name}: {coef:.2f}")

Test set score: 0.8005

Coefficients:
num__area: 47485.93
cat__town_monroe township: -7444.44
cat__town_robinsville: -111.11
cat__town_west windsor: 7555.56


In [124]:
data_to_predict = pd.DataFrame([
    {'area': 3400, 'town': 'monroe township'},
    {'area': 2800, 'town': 'robbinsville'},
    {'area': 3100, 'town': 'west windsor'}
])

predictions_lg = pipe.predict(data_to_predict)
predictions_lg_dummy = pipe_dummy.predict(data_to_predict)
predictions_rf = grid_search.predict(data_to_predict)
predictions_rf_dummy = grid_search_dummy.predict(data_to_predict)

for area, town, lg, lg_dummy, rf, rf_dummy in zip(data_to_predict['area'], data_to_predict['town'], predictions_lg, predictions_lg_dummy, predictions_rf, predictions_rf_dummy):
    print(f"\nFor a {area} sq ft house in {town}:")
    print(f"  - Model 1 (lg) Predicted Price: ${lg:,.2f}")
    print(f"  - Model 2 (lg_dummy) Predicted Price: ${lg_dummy:,.2f}")
    print(f"  - Model 3 (rf) Predicted Price: ${rf:,.2f}")
    print(f"  - Model 4 (rf_dummy) Predicted Price: ${rf_dummy:,.2f}")


For a 3400 sq ft house in monroe township:
  - Model 1 (lg) Predicted Price: $657,166.67
  - Model 2 (lg_dummy) Predicted Price: $657,166.67
  - Model 3 (rf) Predicted Price: $666,000.00
  - Model 4 (rf_dummy) Predicted Price: $664,933.33

For a 2800 sq ft house in robbinsville:
  - Model 1 (lg) Predicted Price: $589,166.67
  - Model 2 (lg_dummy) Predicted Price: $596,611.11
  - Model 3 (rf) Predicted Price: $622,566.67
  - Model 4 (rf_dummy) Predicted Price: $626,866.67

For a 3100 sq ft house in west windsor:
  - Model 1 (lg) Predicted Price: $638,166.67
  - Model 2 (lg_dummy) Predicted Price: $638,166.67
  - Model 3 (rf) Predicted Price: $624,033.33
  - Model 4 (rf_dummy) Predicted Price: $624,066.67


/opt/homebrew/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
